# NLP Assignment 2025/26 - PoliMillionaire QuizBot

**Task.** Build and evaluate local open-weight chatbot models for the online quiz *Who wants to be a PoliMillionaire?*.

**Group members.** TODO: add names and Polimi email addresses.

**Video link.** TODO: add the screen-capture video link.

**Coding assistants statement.** TODO: state which coding assistants, if any, were used.

This notebook uses the official `millionaire_client` package provided with the course. The first version focuses on text mode and compares two local models with a simple ensemble strategy. RAG is intentionally not used in this baseline because the quiz is open-domain, real-time retrieval can be too slow for the 30-second timeout, and a static corpus would be difficult to justify without adding noise.

## 1. Global Settings

All operational settings are centralized here. The rest of the notebook should run top-to-bottom without changing local flags inside later cells.

In [ ]:
# Centralized, all settings are. Scattered through the notebook, they are not.
API_URL = 'http://131.175.15.22:51111/'
USERNAME = 'MarianoAkaMery'
PASSWORD = 'Test1234!'

DRIVE_PROJECT_DIR = '/content/gdrive/MyDrive/[2025-26] - 088946 - NLP PROJECT'
COMPETITION_ID = 1
GAME_MODE = 'text'

MODEL_A_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
MODEL_B_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
USE_ENSEMBLE = True
PREFER_MODEL_B_ON_DISAGREEMENT = True

MAX_LEVELS_TO_PLAY = 15
DELAY_BETWEEN_QUESTIONS_S = 1.0
SAVE_RUN_LOG = True

INSTALL_MODEL_DEPENDENCIES = True
RUN_API_CHECK = True
RUN_MODEL_WARMUP = True
RUN_FULL_GAME = True

print('Settings loaded.')

## 2. Colab Setup

This mounts Google Drive, locates the project folder, and makes the official client package importable.

In [ ]:
# Mounted, Google Drive is. Found, the project folder must be.
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/gdrive/')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

PROJECT_DIR = DRIVE_PROJECT_DIR if IN_COLAB else os.getcwd()
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

print('IN_COLAB:', IN_COLAB)
print('PROJECT_DIR:', PROJECT_DIR)
print('millionaire_client exists:', os.path.isdir(os.path.join(PROJECT_DIR, 'millionaire_client')))

## 3. Dependencies

The API client only needs `requests`, already available in most environments. The local models need `transformers`, `accelerate`, `sentencepiece`, and `torch`.

In [ ]:
# Installed, model dependencies are. Local models, then usable become.
if INSTALL_MODEL_DEPENDENCIES:
    import subprocess
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'transformers',
        'accelerate',
        'sentencepiece',
        'torch'
    ])
    print('Model dependencies installed or already available.')
else:
    print('Dependency installation skipped by settings.')

## 4. Official Game Client

This section logs in and checks the game API using only the course-provided `millionaire_client` package.

In [ ]:
# Imported, the official client is. Invented endpoints, used they are not.
from millionaire_client import MillionaireClient, AuthenticationError, MillionaireError

client = MillionaireClient(API_URL)
user = client.login(USERNAME, PASSWORD)
print(f'Logged in as {user.username} (role={user.role})')

if RUN_API_CHECK:
    competitions = client.competitions.list_all()
    print('\nAvailable competitions:')
    for comp in competitions:
        print(f'  {comp.id}: {comp.name} | max_levels={comp.max_levels} | questions={comp.question_count}')
else:
    print('API check skipped by settings.')

## 5. Prompt and Output Parsing

The game API requires an `option_id`. The model is prompted to output only a numeric id, and the parser accepts only ids that are present in the current question.

In [ ]:
# Built, the prompt is. Parsed, only valid option ids are.
import re

def build_prompt(question):
    options_text = '\n'.join(f'{opt.id}. {opt.text}' for opt in question.options)
    return f"""You are playing a multiple-choice quiz.
Choose the single best answer.
Return only the numeric option id, with no explanation.

Question:
{question.text}

Options:
{options_text}

Answer id:"""

def parse_option_id(raw_text, question):
    valid_ids = {opt.id for opt in question.options}
    numbers = [int(value) for value in re.findall(r'\d+', str(raw_text))]
    for number in numbers:
        if number in valid_ids:
            return number
    return None

## 6. Local Model Wrapper

This wrapper loads an open-weight Hugging Face causal language model locally inside Colab. No LLM API is used.

In [ ]:
# Loaded locally, the model is. Called, no external LLM API is.
import time

class LocalCausalLMAnswerer:
    def __init__(self, model_name, max_new_tokens=8):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.tokenizer = None
        self.model = None

    def load(self):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        if self.model is not None:
            return

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            device_map='auto',
            torch_dtype='auto'
        )
        self.model.eval()

    def answer(self, question):
        self.load()
        prompt = build_prompt(question)
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)

        start = time.time()
        output_ids = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.eos_token_id
        )
        elapsed_s = time.time() - start

        generated_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
        raw = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        return {
            'model': self.model_name,
            'raw': raw,
            'option_id': parse_option_id(raw, question),
            'elapsed_s': elapsed_s,
        }

model_a = LocalCausalLMAnswerer(MODEL_A_NAME)
model_b = LocalCausalLMAnswerer(MODEL_B_NAME)

print('Model A:', MODEL_A_NAME)
print('Model B:', MODEL_B_NAME)

## 7. Model Warm-up

The models must be loaded before starting a real game. If model weights are loaded during a question, the 30-second timer will likely expire.

In [ ]:
# Warmed up, models are. Timed out by loading weights, the game should not be.
if RUN_MODEL_WARMUP:
    print('Loading model A...')
    model_a.load()
    print('Model A ready.')

    if USE_ENSEMBLE:
        print('Loading model B...')
        model_b.load()
        print('Model B ready.')

    print('Warm-up complete.')
else:
    print('Model warm-up skipped by settings.')

## 8. Answer Strategy

The baseline can run either a single model or a two-model ensemble. In the ensemble setting, agreement is accepted; disagreement is resolved by preferring model B, the larger model.

In [ ]:
# Chosen, the final answer is. Simple and explainable, the strategy stays.
def choose_answer(question):
    pred_a = model_a.answer(question)

    if not USE_ENSEMBLE:
        chosen = pred_a['option_id'] if pred_a['option_id'] is not None else question.options[0].id
        return {
            'chosen_option_id': chosen,
            'decision': 'single_model_a',
            'pred_a': pred_a,
            'pred_b': None,
        }

    pred_b = model_b.answer(question)

    if pred_a['option_id'] is not None and pred_a['option_id'] == pred_b['option_id']:
        chosen = pred_a['option_id']
        decision = 'models_agree'
    elif PREFER_MODEL_B_ON_DISAGREEMENT and pred_b['option_id'] is not None:
        chosen = pred_b['option_id']
        decision = 'models_disagree_prefer_b'
    elif pred_a['option_id'] is not None:
        chosen = pred_a['option_id']
        decision = 'models_disagree_prefer_a'
    else:
        chosen = question.options[0].id
        decision = 'fallback_first_option'

    return {
        'chosen_option_id': chosen,
        'decision': decision,
        'pred_a': pred_a,
        'pred_b': pred_b,
    }

## 9. Full Game Run

This cell starts one real text-mode game and submits model answers. The printed output is intentionally compact: level, selected option, decision rule, correctness, and timing.

In [ ]:
# Played, the real game is. Compact, the output remains.
from datetime import UTC, datetime

def play_full_game():
    run_log = []
    game = client.game.start(competition_id=COMPETITION_ID, mode=GAME_MODE)
    print(f'Started session: {game.session_id}')

    while game.in_progress and game.current_level <= MAX_LEVELS_TO_PLAY:
        question = game.current_question
        if question is None:
            break

        time_before_s = game.time_remaining
        decision = choose_answer(question)
        chosen_id = decision['chosen_option_id']
        result = game.answer(chosen_id)

        pred_b_time = decision['pred_b']['elapsed_s'] if decision['pred_b'] else 0.0
        model_time_s = decision['pred_a']['elapsed_s'] + pred_b_time

        print(
            f"Level {game.current_level} | "
            f"chosen={chosen_id} | "
            f"decision={decision['decision']} | "
            f"correct={result.correct} | "
            f"timeout={result.timed_out} | "
            f"model_time={model_time_s:.2f}s | "
            f"earned={result.earned_amount}"
        )

        run_log.append({
            'timestamp': datetime.now(UTC).isoformat(),
            'session_id': game.session_id,
            'level': game.current_level,
            'question_id': question.id,
            'question_text': question.text,
            'options': [{'id': opt.id, 'text': opt.text} for opt in question.options],
            'time_remaining_before_answer_s': time_before_s,
            'chosen_option_id': chosen_id,
            'decision': decision,
            'correct': result.correct,
            'timed_out': result.timed_out,
            'game_over': result.game_over,
            'earned_amount': result.earned_amount,
        })

        if result.game_over or result.timed_out:
            break

        time.sleep(DELAY_BETWEEN_QUESTIONS_S)

    print(f'Final level: {game.current_level}')
    print(f'Final earned amount: {game.earned_amount}')
    return game, run_log

if RUN_FULL_GAME:
    final_game, run_log = play_full_game()
else:
    final_game, run_log = None, []
    print('Full game skipped by settings.')

## 10. Save Results

The log is saved as JSON for later error analysis and for reporting model performance in the final notebook.

In [ ]:
# Saved, the run is. Analyzed later, errors can be.
import json

if SAVE_RUN_LOG and run_log:
    log_dir = Path(PROJECT_DIR) / 'runs'
    log_dir.mkdir(parents=True, exist_ok=True)
    output_path = log_dir / f'run_{datetime.now(UTC).strftime("%Y%m%d_%H%M%S")}.json'
    with output_path.open('w', encoding='utf-8') as f:
        json.dump(run_log, f, indent=2, ensure_ascii=False)
    print('Saved run log:', output_path)
else:
    print('No run log saved.')

## 11. Leaderboard

This retrieves the leaderboard for the selected competition and mode after the run.

In [ ]:
# Checked, the leaderboard is. Compared, our run can be.
leaderboard = client.leaderboard.get(competition_id=COMPETITION_ID, limit=10, mode=GAME_MODE)

print('Leaderboard:', leaderboard.competition.name)
for rank, entry in enumerate(leaderboard.entries, 1):
    marker = ' <-- us' if entry.username == USERNAME else ''
    print(f'{rank}. {entry.username}: {entry.score} | level={entry.reached_level}{marker}')

## 12. Discussion Notes

Points to discuss in the final version:

- Whether the two-model ensemble improves over a single model.
- How often the models disagree.
- Whether larger model B is worth the extra latency.
- Whether answers fit inside the 30-second timeout.
- Which question types produce mistakes.
- Why RAG was excluded from this baseline.
- Why speech mode was not used as the main system: the API already exposes text questions, while speech requires ASR and adds latency.